# Примеры к Лекции 7: Дизайн мультиагентных систем в реальном мире

Два кейса из лекции, реализованные на LangGraph:

1. **Система создания контента** — Supervisor + Reflection loop
2. **Платформа автоматизации кода** — Plan-and-Execute + Reflection loop

Оба кейса демонстрируют принципы из лекции:
- Model tiering (разные модели для разных ролей — в комментариях)
- Декомпозиция по компетенциям
- Ограничение итераций в reflection-циклах
- Реальный поиск через DuckDuckGo, реальное чтение файлов

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import TypedDict, Literal

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from ddgs import DDGS

from llm_config import get_llm, check_api_key

assert check_api_key(), "API key not set"

# Базовый путь для демо-файлов (кейс 2)
DEMO_CODE_DIR = Path(__file__).parent / "_demo_code" if "__file__" in dir() else Path("_demo_code")
print(f"Demo code dir: {DEMO_CODE_DIR}")
print(f"Demo files: {list(DEMO_CODE_DIR.glob('*.py'))}")

In [ ]:
# ============================================================================
# ОБЩИЕ ИНСТРУМЕНТЫ
# ============================================================================

@tool
def web_search(query: str) -> str:
    """Поиск информации в интернете через DuckDuckGo."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
    if not results:
        return f"По запросу '{query}' ничего не найдено."
    return "\n".join(f"- {r['title']}: {r['body']}" for r in results)


# Проверка
print("DuckDuckGo поиск:")
print(web_search.invoke({"query": "multi-agent systems LangGraph 2026"}))

## Кейс 1: Система создания контента

**Задача:** из бриф-документа создать пакет контента — лонгрид и посты для соцсетей.

**Архитектура из лекции:**
```text
Supervisor (координатор)
├── Researcher (поиск фактов)       — модель: дешёвая (GPT-5.4-mini)
├── Writer (лонгрид)                — модель: средняя (Sonnet 4.6)
│   └── Editor (reflection loop)    — модель: дорогая (Opus 4.6)
└── Social Media Agent (посты)      — модель: средняя
```

**Принципы:**
- **Reflection loop**: Writer → Editor → Writer (если нужны правки), max 3 итерации
- **Model tiering**: Researcher — дешёвая, Editor — дорогая (в комментариях)
- **Least privilege**: Writer не имеет инструментов — только данные от Researcher

In [ ]:
# ============================================================================
# КЕЙС 1: СИСТЕМА СОЗДАНИЯ КОНТЕНТА
# ============================================================================

class ContentState(TypedDict):
    brief: str
    research: str
    draft: str
    editor_feedback: str
    revision_count: int
    final_article: str
    social_posts: str
    status: str


# В реальной системе каждый агент использовал бы свою модель:
# researcher_llm = get_llm("gpt-5.4-mini")    # дешёвая — задача простая
# writer_llm = get_llm("claude-sonnet-4.6")    # средняя — генерация текста
# editor_llm = get_llm("claude-opus-4.6")      # дорогая — оценка качества
# Здесь используем одну модель для простоты.
llm = get_llm()


def researcher_node(state: ContentState) -> dict:
    """Исследователь: ищет реальные факты через DuckDuckGo."""
    facts = web_search.invoke({"query": state["brief"]})

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — исследователь. Систематизируй найденные факты по теме. "
            "Выдели 5-7 ключевых тезисов с цифрами и датами. "
            "Только факты — не пиши статью."
        )),
        HumanMessage(content=f"Бриф: {state['brief']}\n\nНайденные данные:\n{facts}"),
    ])
    print(f"  [Researcher] Факты собраны ({len(response.content)} символов)")
    return {"research": response.content, "status": "research_done"}


def writer_node(state: ContentState) -> dict:
    """Писатель: создаёт лонгрид на основе фактов."""
    context = f"Факты от исследователя:\n{state['research']}"
    if state.get("editor_feedback"):
        context += (
            f"\n\nОбратная связь от редактора (итерация {state['revision_count']}):"
            f"\n{state['editor_feedback']}"
            f"\n\nИсправь замечания и улучши статью."
        )

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — технический писатель. Напиши лонгрид (400-600 слов) "
            "на русском языке с заголовком и 3-4 разделами. "
            "Живой, увлекательный стиль. Работай только с предоставленными фактами."
        )),
        HumanMessage(content=context),
    ])
    count = state.get("revision_count", 0)
    if state.get("editor_feedback"):
        count += 1
    print(f"  [Writer] Черновик готов (итерация {count}, {len(response.content)} символов)")
    return {"draft": response.content, "revision_count": count, "status": "draft_ready"}


def editor_node(state: ContentState) -> dict:
    """Редактор: оценивает качество и даёт обратную связь."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — строгий редактор. Оцени статью по критериям:\n"
            "1. Фактическая точность (сверь с исходными фактами)\n"
            "2. Структура и логика изложения\n"
            "3. Стиль и читаемость\n\n"
            "Если статья хорошая — ответь одним словом: APPROVED\n"
            "Если нужны правки — перечисли конкретные замечания."
        )),
        HumanMessage(content=(
            f"Исходные факты:\n{state['research']}\n\n"
            f"Статья:\n{state['draft']}"
        )),
    ])
    feedback = response.content
    is_approved = "APPROVED" in feedback.upper()
    print(f"  [Editor] {'APPROVED' if is_approved else 'Нужны правки'}")
    return {"editor_feedback": feedback, "status": "approved" if is_approved else "needs_revision"}


def should_revise(state: ContentState) -> Literal["writer", "social"]:
    """Условный переход: max 3 итерации reflection."""
    if state["status"] == "approved" or state.get("revision_count", 0) >= 3:
        if state.get("revision_count", 0) >= 3:
            print(f"  [Router] Лимит итераций (3) — переходим дальше")
        return "social"
    return "writer"


def social_media_node(state: ContentState) -> dict:
    """Агент соцсетей: адаптирует контент для разных платформ."""
    article = state["draft"]
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — SMM-специалист. На основе статьи создай:\n"
            "1. Пост для Twitter/X (до 280 символов)\n"
            "2. Пост для LinkedIn (2-3 абзаца)\n"
            "3. Пост для Telegram (краткий)\n\n"
            "Каждый пост — отдельный блок с заголовком платформы."
        )),
        HumanMessage(content=f"Статья:\n{article}"),
    ])
    print(f"  [Social Media] Посты готовы ({len(response.content)} символов)")
    return {"social_posts": response.content, "final_article": article, "status": "completed"}


print("Узлы кейса 1 определены")

In [ ]:
# ============================================================================
# КЕЙС 1: СБОРКА И ЗАПУСК ГРАФА
# ============================================================================

graph = StateGraph(ContentState)

graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_node("editor", editor_node)
graph.add_node("social", social_media_node)

graph.add_edge(START, "researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", "editor")
graph.add_conditional_edges("editor", should_revise)
graph.add_edge("social", END)

app = graph.compile()

print("=== Кейс 1: Система создания контента ===\n")
result = app.invoke({
    "brief": "AI-агенты в 2026: фреймворки, протоколы, тренды",
    "research": "", "draft": "", "editor_feedback": "",
    "revision_count": 0, "final_article": "", "social_posts": "", "status": "",
})

print(f"\n--- Финальная статья ({len(result['final_article'])} символов) ---")
print(result["final_article"][:500] + "...")
print(f"\n--- Посты для соцсетей ---")
print(result["social_posts"][:500] + "...")
print(f"\nИтераций reflection: {result['revision_count']}")

## Кейс 2: Платформа автоматизации кода

**Задача:** по описанию issue найти проблему в коде, написать fix, проверить тестами.

Используем **реальные файлы** из `_demo_code/`:
- `auth.py` — модуль с багом (не проверяет срок действия токена)
- `test_auth.py` — тест, который падает из-за бага

**Архитектура из лекции:**
```text
Planner (планирование)          — модель: дорогая (Opus 4.6)
├── Code Analyst (анализ кода)   — модель: средняя (Sonnet 4.6)
├── Developer (написание fix)    — модель: средняя
├── Tester (запуск тестов)       — модель: средняя
│   └── Developer (reflection loop, max 3 итерации)
└── PR Creator (создание PR)     — модель: дешёвая (Haiku 4.5)
```

In [ ]:
# ============================================================================
# КЕЙС 2: ПЛАТФОРМА АВТОМАТИЗАЦИИ КОДА
# ============================================================================

class CodeState(TypedDict):
    issue: str
    plan: str
    code_analysis: str
    fix: str
    test_result: str
    test_passed: bool
    fix_iterations: int
    pr_description: str
    status: str


# Инструменты — реальное чтение файлов
@tool
def read_file(path: str) -> str:
    """Прочитать файл из репозитория."""
    file_path = DEMO_CODE_DIR / path
    if file_path.exists():
        return file_path.read_text()
    return f"Файл '{path}' не найден в {DEMO_CODE_DIR}"


@tool
def list_files(directory: str) -> str:
    """Список файлов в директории."""
    dir_path = DEMO_CODE_DIR / directory if directory != "." else DEMO_CODE_DIR
    if dir_path.exists():
        files = [f.name for f in dir_path.iterdir() if f.is_file()]
        return "\n".join(files)
    return f"Директория '{directory}' не найдена"


llm = get_llm()


# Проверяем, что файлы на месте
print("Файлы в _demo_code:")
print(list_files.invoke({"directory": "."}))
print("\nauth.py:")
print(read_file.invoke({"path": "auth.py"}))

In [ ]:
# ============================================================================
# КЕЙС 2: УЗЛЫ ГРАФА
# ============================================================================

def planner_node(state: CodeState) -> dict:
    """Planner: создаёт план на основе issue."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — архитектор-планировщик. На основе описания issue:\n"
            "1. Определи, какие файлы нужно изменить\n"
            "2. Опиши шаги для исправления\n"
            "3. Укажи, какие тесты нужно написать/обновить\n"
            "Будь кратким и конкретным."
        )),
        HumanMessage(content=f"Issue: {state['issue']}"),
    ])
    print(f"  [Planner] План готов ({len(response.content)} символов)")
    return {"plan": response.content, "status": "planned"}


def analyst_node(state: CodeState) -> dict:
    """Code Analyst: читает реальные файлы и находит проблему."""
    auth_code = read_file.invoke({"path": "auth.py"})
    test_code = read_file.invoke({"path": "test_auth.py"})

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — аналитик кода. Изучи код и тесты. "
            "Найди причину бага и опиши её. Укажи конкретные строки."
        )),
        HumanMessage(content=(
            f"План:\n{state['plan']}\n\n"
            f"=== auth.py ===\n{auth_code}\n\n"
            f"=== test_auth.py ===\n{test_code}"
        )),
    ])
    print(f"  [Analyst] Анализ завершён ({len(response.content)} символов)")
    return {"code_analysis": response.content, "status": "analyzed"}


def developer_node(state: CodeState) -> dict:
    """Developer: пишет fix. Учитывает feedback от Tester при повторных итерациях."""
    context = f"План:\n{state['plan']}\n\nАнализ:\n{state['code_analysis']}"
    if state.get("test_result") and not state.get("test_passed", True):
        context += (
            f"\n\nТесты ПРОВАЛЕНЫ (итерация {state['fix_iterations']}):\n"
            f"{state['test_result']}\nИсправь код с учётом ошибок тестов."
        )

    # Читаем текущий код
    current_code = read_file.invoke({"path": "auth.py"})
    context += f"\n\nТекущий код auth.py:\n{current_code}"

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — разработчик. Напиши исправленный код для auth.py. "
            "Включи проверку срока действия токена. "
            "Покажи полный исправленный файл."
        )),
        HumanMessage(content=context),
    ])
    iterations = state.get("fix_iterations", 0)
    if state.get("test_result") and not state.get("test_passed", True):
        iterations += 1
    print(f"  [Developer] Fix написан (итерация {iterations})")
    return {"fix": response.content, "fix_iterations": iterations, "status": "fix_ready"}


def tester_node(state: CodeState) -> dict:
    """Tester: оценивает fix."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — тестировщик. Оцени предложенный fix:\n"
            "1. Решает ли он проблему из issue?\n"
            "2. Не ломает ли существующую функциональность?\n"
            "3. Есть ли edge cases?\n\n"
            "Если fix корректен — ответь: TESTS PASSED\n"
            "Если есть проблемы — опиши их."
        )),
        HumanMessage(content=(
            f"Issue: {state['issue']}\n\n"
            f"Анализ: {state['code_analysis']}\n\n"
            f"Fix:\n{state['fix']}"
        )),
    ])
    passed = "PASSED" in response.content.upper()
    print(f"  [Tester] {'TESTS PASSED' if passed else 'TESTS FAILED'}")
    return {"test_result": response.content, "test_passed": passed, "status": "tested"}


def should_retry_fix(state: CodeState) -> Literal["developer", "pr_creator"]:
    """Условный переход: max 3 итерации."""
    if state.get("test_passed", False):
        return "pr_creator"
    if state.get("fix_iterations", 0) >= 3:
        print(f"  [Router] Лимит итераций (3) — создаём PR с текущим fix")
        return "pr_creator"
    return "developer"


def pr_creator_node(state: CodeState) -> dict:
    """PR Creator: генерирует описание PR."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — автор PR. Сгенерируй описание Pull Request в Markdown:\n"
            "- Заголовок (до 70 символов)\n"
            "- Описание: что исправлено и почему\n"
            "- Тест-план: как проверить"
        )),
        HumanMessage(content=(
            f"Issue: {state['issue']}\n\n"
            f"Fix:\n{state['fix']}\n\n"
            f"Тесты: {'прошли' if state.get('test_passed') else 'есть проблемы'}"
        )),
    ])
    print(f"  [PR Creator] Описание PR готово")
    return {"pr_description": response.content, "status": "pr_created"}


print("Узлы кейса 2 определены")

In [ ]:
# ============================================================================
# КЕЙС 2: СБОРКА И ЗАПУСК ГРАФА
# ============================================================================

graph = StateGraph(CodeState)

graph.add_node("planner", planner_node)
graph.add_node("analyst", analyst_node)
graph.add_node("developer", developer_node)
graph.add_node("tester", tester_node)
graph.add_node("pr_creator", pr_creator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "analyst")
graph.add_edge("analyst", "developer")
graph.add_edge("developer", "tester")
graph.add_conditional_edges("tester", should_retry_fix)
graph.add_edge("pr_creator", END)

app = graph.compile()

print("=== Кейс 2: Платформа автоматизации кода ===\n")
result = app.invoke({
    "issue": (
        "Баг: функция authenticate() в auth.py не проверяет срок действия токена. "
        "Expired токены принимаются как валидные. Тест test_expired_token падает."
    ),
    "plan": "", "code_analysis": "", "fix": "", "test_result": "",
    "test_passed": False, "fix_iterations": 0, "pr_description": "", "status": "",
})

print(f"\n--- PR Description ---")
print(result["pr_description"])
print(f"\nИтераций fix: {result['fix_iterations']}")
print(f"Тесты прошли: {result['test_passed']}")

## Итоги: чеклист проектирования

Перед созданием MAS пройдите 8 вопросов из лекции:

| # | Вопрос | Кейс 1 (контент) | Кейс 2 (код) |
|---|--------|-------------------|---------------|
| 1 | Нужна ли MAS? | Да: 3 из 5 условий | Да: компетенции + верификация |
| 2 | Декомпозиция? | По компетенциям | По компетенциям + по этапам |
| 3 | Паттерн? | Supervisor + Reflection | Plan-and-Execute + Reflection |
| 4 | Уровни иерархии? | 1 (supervisor + workers) | 1 (planner + workers) |
| 5 | State vs messages? | Shared state (одна команда) | Shared state (одна команда) |
| 6 | Модели? | Tiering: mini/sonnet/opus | Tiering: opus/sonnet/haiku |
| 7 | Безопасность? | Writer не имеет инструментов | Tester: только shell, не filesystem |
| 8 | Тестирование? | Каждый агент изолированно + e2e | Каждый агент + integration tests |